In [1]:
print("hello")

hello


In [2]:
%pwd

'c:\\Users\\Simaak\\Documents\\InternProjects\\Medical Chatbot\\ChatBot\\research'

In [3]:
import os
os.chdir(r'c:\Users\Simaak\Documents\InternProjects\Medical Chatbot\ChatBot')

In [4]:
%pwd

'c:\\Users\\Simaak\\Documents\\InternProjects\\Medical Chatbot\\ChatBot'

In [5]:
%pip install langchain langchain-community langchain-text-splitters langchain-pinecone langchain-google-genai pypdf

Note: you may need to restart the kernel to use updated packages.


In [6]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.documents import Document

c:\Users\Simaak\anaconda3\envs\medibot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [8]:
# Cell 8: Load PDFs
extracted_data = load_pdf_files("data")
print(f"Loaded {len(extracted_data)} pages from PDFs")

Loaded 637 pages from PDFs


In [9]:
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(Document(page_content=doc.page_content, metadata={"source": src}))
    return minimal_docs

minimal_docs = filter_to_minimal_docs(extracted_data)

In [10]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
        length_function=len
    )
    text_chunks = text_splitter.split_documents(minimal_docs)
    return text_chunks

texts_chunks = text_split(minimal_docs)
print(f"No of chunks: {len(texts_chunks)}")

No of chunks: 5859


In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embedding = download_embeddings()

C:\Users\Simaak\AppData\Local\Temp\ipykernel_15504\1359395703.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)


In [12]:
vector = embedding.embed_query("Hello world")
print("Vector length:", len(vector))

Vector length: 384


In [13]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [14]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ["HUGGINGFACE_API_KEY"] = HUGGINGFACE_API_KEY

In [15]:
from pinecone import Pinecone, ServerlessSpec

pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)

In [16]:
index_name = "medical-chatbot"

# Check if index exists
if index_name not in pc.list_indexes().names():
    print(f"Creating new index: {index_name}")
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created successfully!")
else:
    print(f"Index '{index_name}' already exists. Skipping creation.")

index = pc.Index(index_name)

Index 'medical-chatbot' already exists. Skipping creation.


In [17]:
from langchain_pinecone import PineconeVectorStore

index_stats = index.describe_index_stats()
total_vectors = index_stats.get('total_vector_count', 0)

if total_vectors == 0:
    print("Index is empty. Uploading documents...")
    docsearch = PineconeVectorStore.from_documents(
        documents=texts_chunks, 
        embedding=embedding, 
        index_name=index_name
    )
    print(f"Successfully uploaded {len(texts_chunks)} document chunks to Pinecone!")
else:
    print(f"Index already contains {total_vectors} vectors. Skipping upload to avoid duplicates.")
    print("Loading existing index...")
    docsearch = PineconeVectorStore.from_existing_index(
        index_name=index_name,
        embedding=embedding
    )
    print("Connected to existing index.")

Index already contains 5859 vectors. Skipping upload to avoid duplicates.
Loading existing index...
Connected to existing index.


In [18]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)
print("Loaded existing Pinecone index")

Loaded existing Pinecone index


In [19]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})


In [20]:
retrieved_docs = retriever.invoke("What is Acne?")
print(f"Retrieved {len(retrieved_docs)} documents")
for i, doc in enumerate(retrieved_docs):
    print(f"\nDoc {i+1}: {doc.page_content[:200]}...")

Retrieved 3 documents

Doc 1: GALE ENCYCLOPEDIA OF MEDICINE 226
Acne
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26...

Doc 2: GALE ENCYCLOPEDIA OF MEDICINE 2 25
Acne
Acne vulgaris affecting a woman’s face. Acne is the general
name given to a skin disorder in which the sebaceous
glands become inflamed. (Photograph by Biophoto...

Doc 3: Acidosis see Respiratory acidosis; Renal
tubular acidosis; Metabolic acidosis
Acne
Definition
Acne is a common skin disease characterized by
pimples on the face, chest, and back. It occurs when the
po...


In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp", temperature=0)

In [22]:
%pip install --upgrade langchain langchain-core langchain-community

Note: you may need to restart the kernel to use updated packages.


In [29]:
# Complete working RAG chain using LCEL (LangChain Expression Language)
# Run this cell after you've already defined 'retriever' and 'chatModel'

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Helper function to format retrieved documents
def format_docs(docs):
    """Format a list of documents into a single string"""
    return "\n\n".join([doc.page_content for doc in docs])

# Define the prompt template
prompt_template = """You are a Medical expert assistant. Provide accurate and concise information based on the provided documents.

Context:
{context}

Question: {question}

Answer: Provide a clear, concise answer using the context above. If you don't know the answer based on the context, say so. Keep your answer to 3 sentences maximum."""

prompt = ChatPromptTemplate.from_template(prompt_template)

# Create the RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | chatModel
    | StrOutputParser()
)

# Function to ask questions with source tracking
def ask_medical_question(question):
    """Ask a question and get answer with sources"""
    # Get the answer
    answer = rag_chain.invoke(question)
    
    # Get the source documents
    source_docs = retriever.invoke(question)
    
    print(f"Question: {question}\n")
    print(f"Answer: {answer}\n")
    print("Sources:")
    for i, doc in enumerate(source_docs, 1):
        source = doc.metadata.get('source', 'Unknown')
        preview = doc.page_content[:100].replace('\n', ' ')
        print(f"{i}. {source}")
        print(f"   Preview: {preview}...\n")
    
    return answer

# Test the function
print("="*80)
print("Testing Medical Chatbot")
print("="*80 + "\n")

# Test question 1
ask_medical_question("What is Down Syndrome?")

print("\n" + "="*80 + "\n")

# Test question 2
ask_medical_question("What are the symptoms of diabetes?")

print("\n" + "="*80 + "\n")

# Test question 3
ask_medical_question("What is autism?")
ask_medical_question("What are the treatments for autism?")

Testing Medical Chatbot

Question: What is Down Syndrome?

Answer: Down syndrome, also known as trisomy 21, usually results from having three copies of chromosome 21 instead of the usual two. Abnormally low levels of AFP may indicate an increased risk of the fetus having Down syndrome. It is a condition that includes mental retardation and a distinctive physical appearance.

Sources:
1. data\Medical_book.pdf
   Preview: contain three copies of certain chromosomes rather than the usual two. Down syndrome, or trisomy 21,...

2. data\Medical_book.pdf
   Preview: Levels may also be high if there is too little fluid in the amniotic sac around the fetus, more than...

3. data\Medical_book.pdf
   Preview: Saunders Co., 1996. Lott, Judy Wright. “Fetal Development: Environmental Influ- ences and Critical P...



Question: What are the symptoms of diabetes?

Answer: The symptoms of Type I diabetes mellitus include fatigue and abnormally high blood glucose levels (hyperglycemia). If left untreate

'There is no cure for autism, but treatments aim to reduce specific symptoms through a spectrum of interventions. These interventions include training in music, listening, vision, speech and language, and senses. Special diets and medications may also be prescribed.'

In [24]:
# response = rag_chain.invoke({"input": "What is Down Syndrome?"})
# print("Answer:", response["answer"])
# print("\nSources:")
# for doc in response["context"]:
#     print(f"- {doc.metadata.get('source', 'Unknown')}")

In [25]:
# def ask_medical_question(question):
#     """Ask a question to the medical chatbot"""
#     response = rag_chain.invoke({"input": question})
#     return response["answer"]

In [26]:
# # Example usage:
# answer = ask_medical_question("What are the symptoms of diabetes?")
# print(answer)